# Welcome to Aminoacid ! 🧬🎉

Welcome to **aminoacid**, an interactive web app designed to easily learn and test your skills on the different aminoacids.

The main functionalities of this python package will be explained in the following, but first...

🔨 **Let's import everything !** 

To import all the files needed, run the following code:

In [ ]:
import streamlit as st
from streamlit_ketcher import st_ketcher 
from rdkit import Chem 
from rdkit.Chem import Draw 
import random 
import base64
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..", "..")))
from data.aminoacidlist import amino_acids
from data.aminoacidlist import aa_stereo
from pathlib import Path

Now that we are all set up, let's dive in ! 🏊‍♂️ 

# Introduction 📚

Amino acids, which make up proteins, are essential for the proper functioning of life. They help repair tissues, absorb nutrients and participate in the transport of neurotransmitters and so on.
There are 20 amino acids, all composed of a carboxyl group, an amino group, a hydrogen and a side chain that differentiates them. 

That being said, it is important to know about them and their structures, which give us an idea of their chemical properties.
It can be difficult to learn these 20 different structures. That's why we have decided to create a platform where you can learn them simply using flashcards in a grid layout and test your knowledge by drawing and naming them !

# Material and methods 

## 1. Learn amino acids 📖

**Goal**: Learn the structure of amino acids

Let's learn the amino acids !

In this part the function **show_learn()** will be explained. 

### 1.1 Informations about amino acids

A quick review and image of the common structure of amino acids are displayed thanks to streamlit API functions. With st.caption text in small font is shown. st.write is used to displays text and st.image displays an image from a file.  

### 1.2 The visualisation of the amino acids 

A grid layout of buttons is displayed. Once clicked the button shows or hides the image of the corresponding amino aicd.

<img src="../assets/learn_grid.png" width="600"> 

To keep the image once the button is clicked, it is necessary to have it in a specific session as streamlit reruns everything each time the user interacts with the app. 
The state "visible" is intialized with a key for every element in the dictionnary aa_stereo, initally all set to False to be hidden. 

The number of columns per row is set at 4. And the dictionnary aa_stereo is converted into a list of tuples (name, smiles) for iteration. 

The loop enables the creation of a grid by inserting column containers in a row.
Each row is filled with buttons and it's image. For every amino acid a button is shown with its name. When the button is clicked the visiblity is toggled. If the session state visible[name] is true, the smiles is converted to a molecule and the molecule is drawn as an image using RDkit. This image is then displayed with st.image(). 

## 2. Game 🎮

Now that you master your amino acids, let's test your knowledge with an interactive game ! 

### 2.1 Draw amino acids 🖍

**Goal:** draw the amino acid asked for and receive an instantaneous feedback.
Keep track of your progression thanks to the progression bar, and retry your mistakes only until you are officially an amino acid expert 🎉

#### Initialization and state management

As Streamlit reruns the whole code everytime the user interacts with the app, it is of crucial importance to make sure that the actual state is maintained throughout the session. As an example, the user's score should not undergo a reset everytime a button is pressed.
This is why are first defined all keys necessary to a fluid functioning of the drawing part of the quiz through *st.session_state* command.

The keys initialized are:
- *retry_mode*: to differentiate the first round from retrying incorrect answers.
- *incorrect_answers*: to store all incorrect answers for future retry (in a list).
- *round_order*: to define the sequence of questions **randomly**. If in retrymode, only the incorrect answers are taken into account; otherwise all 20 amino acids are included.
- *ketcher_key*: to refresh the Ketcher interface where structures are drawn between each question.
- *current_index*: to keep track of the current question position. It is incremented everytime the next question is reached.
- *score*: to count the number of correct answers.
- *answered* and *show_score_clicked*: to control single answer and differentiate the quizz interface from the result screen.

#### Question and progress bar

Both question and progress bar are here to guide the user through the quiz. 

The current amino acid is displayed with command *markdown* and updated at each new question according to the *round_order* sequence.

As for the progress bar, it appears above the drawing interface. Progress is calculated as ratio of the current question index over total number of questions.

<img src="../assets/Draw-interface.PNG" width="600"> 

#### Ketcher drawing interface

The drawing interface is central to this game mode. The Ketcher chemical structure editor is used for its high performance and adaptability through *st_ketcher* command. When the user is satisfied with his answer and clicks on "Apply", is generated the SMILES string corresponding to the structure drawn. 

The function *are_equivalent* checks if the user's structure is equivalent to the correct one. In order to do so, the SMILES strings are converted to non-ambiguous InChI representation (RDKit library). Indeed, different SMILES versions exist for the same compound, while a unique canonical InChI exists for a given compound.

In [ ]:
def are_equivalent(smiles1, smiles2):
    mol1 = Chem.MolFromSmiles(smiles1)
    mol2 = Chem.MolFromSmiles(smiles2)
    if mol1 is None or mol2 is None:
        return False
    return Chem.MolToInchi(mol1) == Chem.MolToInchi(mol2)

A success message is displayed if the structures are equivalent.
However if the molecules don't match, an error message and an image of the correct structure are shown.

<img src="../assets/Draw-false.PNG" width="600"> 

#### Scoring system

The main issue here is to make sure that the answer is considered correct only at the first attempt.

On the one hand, the score is incremented only if key *answered* is *False*:

In [ ]:
if are_equivalent(ketcher_smiles, target_smiles):
    if not st.session_state.answered:
        st.session_state.score += 1
        st.session_state.answered = True

On the other hand, it is made sure that questions can be answered only once or a message indicating to click onto "Next" will appear:

<img src="../assets/Draw-already-answered.PNG" width="600"> 

At the end of the round, the score is displayed after clicking on the "Show Score" button. According to the score percentage is proposed a different message.

#### Retry mistakes option

When the round has been finished, it is possible to retry only the amino acids drawn incorrectly. This is made possible by the *incorrect_answers* key defined previously. It is a list that stores all amino acids drawn wrong. 

If this option is chosen, the retry round will work exactly the same way as the first round, only with less amino acids. All keys are reset and the *retry_mode* is activated. To be noted that the cycle is not limited, meaning that the user can retry until every answer is right.

### 2.2 Name amino acids

**Goal** : This section allows you to practise naming the different amino acids by looking at their structure. It was created using the **name_quizz()** function. 

As for the other sections, the representations of the amino acids are obtained using smiles and grouped together in a dictionnary, **amino_acids**. It is imported from the file *aminoacidlist*.

### Initialization and session state

As for the previous part, the command *session.state* has been used to keep the actual state throughout the whole session. 
The keys are for :

- *name_quizz_order* : to create the order of amino acids **randomly**.
- *name_quizz_index* : to keep track of the question position.
- *wrong_answer* : is a list storing the wrong answers.
- *name_quizz_score* : to track the number of correct answers.
- *user_guess_input* : to initialize the user's input once.
- *reset_input_flag* : to reset the input field
- *name_quizz_answered* : is a boolean flag to see if the user has already answered the question.

### Naming interface

After entering the name of the amino acid, press the ‘Check answer’ button and you'll find out whether the answer given is correct or not. 

To avoid having problems with the way the answer is written (upper, lower case), the *.lower()* function was used. 

In [ ]:
if st.button("Check answer") and not st.session_state.name_quizz_answered:
    if user_guess.lower() == correct_name.lower():
        st.success("✅ Correct!")
        st.session_state.name_quizz_score += 1
    else:
        st.error(f"❌ Nope! The correct answer was: **{correct_name}**")
        st.session_state.wrong_answers.append(correct_name)
    st.session_state.name_quizz_answered = True

When the answer is wrong, the correct answer is marked below as following:

<img src="../assets/wrong_answer.png" width="600"> 

There were problems when, after entering the name of the amino acid and clicking on the new molecule, the name of the previous amino acid remained written. This part has been controlled with the following codes:

In [ ]:
default_value = "" if st.session_state.reset_input_flag else st.session_state.get("user_guess_input", "")
user_guess = st.text_input("Enter the name of this amino acid:", value=default_value, key="user_guess_input").strip()

if st.session_state.reset_input_flag:  
    st.session_state.reset_input_flag = False

The first line is used to defined the *default_value* variable. If *reset_input_flag* is True, the input will appear cleared, otherwise the output is preserved across reruns. The last two lines allow to turn off the *reset_input_flag*.

### The score

To keep track of the number of wrong answers, a list *wrong_answer* has been created with *session_state*. 

Several cases had to be treated. Firstly, if the user did not check their answer and clicked directly on the *Next molecule* button.

Secondly, to be sure that the user can only answer once. Otherwise, a message will appear telling the user to press the *Next* button.

<img src="../assets/error_message.png" width="600"> 

Finally, once all the amino acids have been made, the final score will be written, along with various messages depending on the score obtained.

### Retry wrong amino acids

After finishing the whole set, a button *Retry incorrect answers* appear allowing the questions to be repeated with the wrong amino acid answers. To do so, a new interface should be set up, in particular by activating *retry_mode*, which has been used to differentiate if it's the first round or not. But otherwise the game works as before.

In [ ]:
if st.button("🔄 Retry incorrect answers "):
    st.session_state.name_quizz_order = [(aa, amino_acids[aa]) for aa in st.session_state.wrong_answers]
    st.session_state.name_quizz_index = 0
    st.session_state.name_quizz_score = 0
    st.session_state.wrong_answers = []
    st.session_state.reset_input_flag = False
    st.session_state.name_quizz_answered = False
    st.session_state.retry_mode = True 
    st.rerun()

For this to work properly, several things had to be changed, such as the final score. Depending on the game mode, the number of total questions had to be adapted. 

### Reset sessions

When finishing the game, to reset the session, the **reset_quizz** function is used. If there is no input, which means *quizz_variables* is False, it takes a list by default, which contains all the session_state variables. 

In [ ]:
def reset_quizz(quizz_variables=[]):
    quizz_variables = quizz_variables or [
        'name_quizz_order',
        'wrong_answers',
        'reset_input_flag',
        'name_quizz_score',
        'name_quizz_index',
        'user_guess_input',
        'name_quizz_answered'
        ]

# Tests 

The coverage of the code by tests is low at 14%. This is due to the fact that we only have two functions with an input and output, that can be easily tested. All other functions (set_background(), show_menu(), draw_quizz(), reset_quizz(), name_quizz(), show_quizz(), show_learn()) are functions running with streamlit and therefore cannot be tested like the two other functions, are_equivalent() and get_base64(). However, by running aa_app.py with streamlit one can see that the functions work well. 

The function **set_background()** works because the backround is indeed the chosen image, see screenshot of the app. **show_menu** functions as well with two buttons being displayed which makes you go to their page. The function **show_quizz()** functions as it allows you to choose between two game modes. 

<img src="../assets/menu_streamlit.png" width="600">                  <img src="../assets/show_quizz_streamlit.png" width="250"> 

# Limitations

1. **Light/Dark mode**: Depending on whether the user's computer is in light or dark mode, the interface and text colors are different. Indeed, in dark mode the text is written in white and the web app becomes less readable. However this being a personal choice remaining to the user, it cannot be dealt with.

2. **Rerun of full script**: As Streamlit reruns the entire code every time a user interacts with a widget (e.g. clicking on a button), noticeable delays and performance issues can arise especially as the script becomes more complex.


3. **Timer**: The timer and 'Already answered' message turned out to be incompatible, even with diverse methods tried (*st_autorefresh()*, *time.sleep(1)* + *st.rerun()*).

 # Challenges 

1. **The tests:** One of the biggest problems in testing the functions is the fact that the application was created using streamlit. In addition, very few functions in the application have a return, which further limited the number of tests that could be carried out.

2. **Retry in the draw part:** The retry part was added afterward, which made it complicated to find where to add the code in each part to make it work.

3. **Handling conflict in Github:** Avoiding conflicts on Github was also a challenge, even though each of us worked on our own branch, modifying the same lines of code still produced merge conflicts. To avoid these conflicts, we made sure to pull everyone else’s changes before making our own.

4. **Problems with tox:** It was complicated to learn how to use tox. 
    

# Conclusion

Despite the limitations encountered, **Aminoacid** offers a fun way to learn the structures of amino acids and be able to differentiate between them. 

To make the game even better, other features can be added, such as a stopwatch to challenge yourself on the speed of your answers. Or adding difficulty levels based on the number of questions to answer or the difficulty of the answers. Then, to learn more about amino acids, there could be functionalities to find out which ones are hydrophilic, to know their pKa so as to be able to draw their protonated/deprotonated structure.